In [1]:
import boto3
import pandas as pd
import os
import chardet
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from typing import Tuple

In [ ]:
import boto3
import os
import zipfile
import pandas as pd
import shutil
import os

# AWS S3 credentials
ACCESS_KEY = '' # Replace with ACCESS_KEY
SECRET_KEY = '' # Replace with SECRET_KEY
BUCKET_NAME = 'anyoneai-datasets'
PREFIX = 'credit-data-2010/'

# Usar cwd en Jupyter para definir carpeta destino
DOWNLOAD_DIR = os.path.join(os.getcwd(), 'dataset', 'credit_data_2010')

# Eliminar carpeta si existe
if os.path.exists(DOWNLOAD_DIR):
    print(f"⚠️ Folder {DOWNLOAD_DIR} already exists. Deleting it...")
    shutil.rmtree(DOWNLOAD_DIR)

# Crear la carpeta nuevamente
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Conectar a S3
s3 = boto3.client(
    's3',
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY
)

# Listar archivos con el prefijo dado
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=PREFIX)

# Descargar archivos
print("Downloading files from S3...")
for obj in response.get('Contents', []):
    file_key = obj['Key']
    file_name = file_key.split('/')[-1]
    if not file_name:
        continue  # saltar carpetas

    dest_path = os.path.join(DOWNLOAD_DIR, file_name)
    temp_dest_path = dest_path + ".temp"

    print(f"→ Downloading: {file_name}")
    try:
        s3.download_file(BUCKET_NAME, file_key, temp_dest_path)

        if os.path.exists(dest_path):
            os.remove(dest_path)
        os.rename(temp_dest_path, dest_path)
    except (PermissionError, FileExistsError) as e:
        print(f"⚠️ Error downloading {file_name}: {e}")
        continue

    # Si es .zip, extraer
    if file_name.endswith('.zip') and os.path.exists(dest_path):
        print(f"→ Extracting: {file_name}")
        try:
            with zipfile.ZipFile(dest_path, 'r') as zip_ref:
                zip_ref.extractall(DOWNLOAD_DIR)
            os.remove(dest_path)
        except Exception as e:
            print(f"⚠️ Error extracting {file_name}: {e}")

# Convertir a .csv si no lo son (sin borrar los archivos originales)
print("\nConverting files to .csv format...")
for root, _, files in os.walk(DOWNLOAD_DIR):
    for file in files:
        file_path = os.path.join(root, file)
        file_lower = file.lower()

        if not file_lower.endswith('.csv'):
            try:
                new_file_path = os.path.splitext(file_path)[0] + '.csv'

                if file_lower.endswith('.xls') or file_lower.endswith('.xlsx'):
                    print(f"→ Converting Excel to CSV: {file}")
                    df = pd.read_excel(file_path)

                elif file_lower.endswith('.txt'):
                    print(f"→ Converting TXT to CSV: {file}")
                    df = None
                    try:
                        df = pd.read_csv(file_path, delimiter='\t', encoding='utf-8', low_memory=False)
                        print(f"  Successfully read {file} with UTF-8")
                    except UnicodeDecodeError:
                        print(f"  ⚠️ UTF-8 failed. Trying latin1...")
                        try:
                            df = pd.read_csv(file_path, delimiter='\t', encoding='latin1', low_memory=False)
                            print(f"  Successfully read {file} with latin1")
                        except Exception as e_latin:
                            print(f"  ❌ Failed with latin1: {e_latin}")
                            continue
                    except Exception as e_utf8:
                        print(f"  ❌ Other error reading {file}: {e_utf8}")
                        continue

                    if df is None:
                        continue

                else:
                    print(f"→ Skipping unsupported file: {file}")
                    continue

                if df is not None:
                    df.to_csv(new_file_path, index=False)
                    print(f"✅ Saved as CSV: {os.path.basename(new_file_path)} (original kept)")
            except Exception as e:
                print(f"⚠️ Error processing {file}: {e}")

print("\n✅ All files processed and converted to .csv in:", DOWNLOAD_DIR)


⚠️ Folder c:\Users\santi\Desktop\FINALRISK\Creda3\Creda\dataset\credit_data_2010 already exists. Deleting it...


ClientError: An error occurred (InvalidAccessKeyId) when calling the ListObjectsV2 operation: The AWS Access Key Id you provided does not exist in our records.

FUNCTION TO TRANSFORM TO CSV

In [13]:
def get_datasets() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Load all necessary datasets for the Credit Risk PAKDD 2010 project.

    Returns:
        modeling_data : pd.DataFrame
            Main training dataset

        prediction_data : pd.DataFrame
            Dataset for prediction (test set)

        leaderboard_data : pd.DataFrame
            Dataset used for leaderboard evaluations

        submission_example : pd.DataFrame
            Example format for submission file

        variables_description : pd.DataFrame
            Description of the features (from .XLS)
    """
    base_path = "dataset/credit_data_2010"

    modeling_data = pd.read_csv(os.path.join(base_path, "PAKDD2010_Modeling_Data.csv"))
    prediction_data = pd.read_csv(os.path.join(base_path, "PAKDD2010_Prediction_Data.csv"))
    leaderboard_data = pd.read_csv(os.path.join(base_path, "PAKDD2010_Leaderboard_Data.csv"))
    submission_example = pd.read_csv(os.path.join(base_path, "PAKDD2010_Leaderboard_Submission_Example.csv"))
    variables_description = pd.read_excel(os.path.join(base_path, "PAKDD2010_VariablesList.XLS"))

    return modeling_data, prediction_data, leaderboard_data, submission_example, variables_description

In [14]:
modeling_data, prediction_data, leaderboard_data, submission_example, variables_description = get_datasets()

column_names = variables_description["Var_Title"].tolist()
column_names_no_target = column_names[:-1]

modeling_data.columns = column_names
prediction_data.columns = column_names_no_target
leaderboard_data.columns = column_names_no_target

C:\Users\santi\AppData\Local\Temp\ipykernel_13488\1858322663.py:23: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  modeling_data = pd.read_csv(os.path.join(base_path, "PAKDD2010_Modeling_Data.csv"))


In [15]:
modeling_data

,ID_CLIENT,CLERK_TYPE,PAYMENT_DAY,APPLICATION_SUBMISSION_TYPE,QUANT_ADDITIONAL_CARDS,POSTAL_ADDRESS_TYPE,SEX,MARITAL_STATUS,QUANT_DEPENDANTS,EDUCATION_LEVEL,...,FLAG_HOME_ADDRESS_DOCUMENT,FLAG_RG,FLAG_CPF,FLAG_INCOME_PROOF,PRODUCT,FLAG_ACSP_RECORD,AGE,RESIDENCIAL_ZIP_3,PROFESSIONAL_ZIP_3,TARGET_LABEL_BAD=1
0,2,C,15,Carga,0,1,F,2,0,0,...,0,0,0,0,1,N,34,230,230,1
1,3,C,5,Web,0,1,F,2,0,0,...,0,0,0,0,1,N,27,591,591,0
2,4,C,20,Web,0,1,F,2,0,0,...,0,0,0,0,1,N,61,545,545,0
3,5,C,10,Web,0,1,M,2,0,0,...,0,0,0,0,1,N,48,235,235,1
4,6,C,10,0,0,1,M,2,0,0,...,0,0,0,0,2,N,40,371,371,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49994,49996,C,10,0,0,1,F,1,2,0,...,0,0,0,0,1,N,36,591,591,1
49995,49997,C,25,0,0,1,F,1,0,0,...,0,0,0,0,2,N,21,186,186,0
49996,49998,C,5,Web,0,1,M,2,3,0,...,0,0,0,0,1,N,41,715,715,0
49997,49999,C,1,Web,0,1,F,1,1,0,...,0,0,0,0,1,N,28,320,320,1


In [16]:
def data_preparation(pre_data: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare the data for modeling.

    Args:
        modeling_data (pd.DataFrame): The main dataset for training.
        prediction_data (pd.DataFrame): The dataset for prediction.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Prepared training and testing datasets.
    """
    post_data = pre_data.copy()

    post_data['HAS_CREDIT_CARD'] = post_data[['FLAG_VISA', 'FLAG_MASTERCARD', 'FLAG_DINERS', 
                                'FLAG_AMERICAN_EXPRESS', 'FLAG_OTHER_CARDS']].max(axis=1)

    cols_to_lower = ["CITY_OF_BIRTH", "RESIDENCIAL_CITY", "RESIDENCIAL_BOROUGH", "PROFESSIONAL_CITY", "PROFESSIONAL_BOROUGH"]

    for col in cols_to_lower:
        if col in modeling_data.columns:
         post_data[col] = post_data[col].apply(
            lambda x: x.lower() if isinstance(x, str) else x
        )

    def replace_pseudo_missing(df: pd.DataFrame, targets: list = ["0", "", " ", "NaN", "None"]) -> pd.DataFrame:
        df = df.copy()
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
                df[col] = df[col].replace(targets, np.nan)
            elif pd.api.types.is_numeric_dtype(df[col]):
                # Solo reemplazar 0 por NaN si es común (pseudo-missing)
                zero_ratio = (df[col] == 0).sum() / len(df)
                if zero_ratio > 0.99:
                    df[col] = df[col].replace(0, np.nan)
        return df    
    
    post_data = replace_pseudo_missing(post_data)

    def group_top_categories(df: pd.DataFrame, col: str, top_k: int = 100, other_label: str = "otros") -> pd.Series:
        """
        Agrupa las categorías menos frecuentes en una categoría 'otros', dejando solo las top_k más comunes.

        Convierte la columna a string para asegurar la comparación.

        Parameters:
            df (pd.DataFrame): DataFrame de entrada
            col (str): Nombre de la columna categórica o numérica con alta cardinalidad
            top_k (int): Número de categorías más frecuentes a mantener
            other_label (str): Etiqueta para agrupar el resto

        Returns:
            pd.Series: Serie con valores agrupados
        """
        col_as_str = df[col].astype(str)
        top_values = col_as_str.value_counts().nlargest(top_k).index
        return col_as_str.apply(lambda x: x if x in top_values else other_label)
    
    post_data["RESIDENCIAL_CITY"] = group_top_categories(post_data, "RESIDENCIAL_CITY", top_k=45)
    post_data["RESIDENCIAL_ZIP_3"] = group_top_categories(post_data, "RESIDENCIAL_ZIP_3", top_k=45)
    post_data["RESIDENCIAL_PHONE_AREA_CODE"] = group_top_categories(post_data, "RESIDENCIAL_PHONE_AREA_CODE", top_k=45)
    post_data["CITY_OF_BIRTH"] = group_top_categories(post_data, "CITY_OF_BIRTH", top_k=45)

    post_data["TOTAL_MONTHLY_INCOME"] = post_data["PERSONAL_MONTHLY_INCOME"] + post_data["OTHER_INCOMES"]

    post_data = post_data[post_data["QUANT_DEPENDANTS"] < 50]

    post_data['FLAG_OTHER_CARDS'] = post_data['FLAG_OTHER_CARDS'] + post_data['FLAG_AMERICAN_EXPRESS']

    selected_features_final = [
    'PAYMENT_DAY',
    'APPLICATION_SUBMISSION_TYPE',
    'SEX',
    'MARITAL_STATUS',
    'QUANT_DEPENDANTS',               
    'STATE_OF_BIRTH',
    'CITY_OF_BIRTH',                  
    'RESIDENCIAL_STATE',
    'RESIDENCIAL_CITY',               
    'FLAG_RESIDENCIAL_PHONE',
    'RESIDENCE_TYPE',                 
    'MONTHS_IN_RESIDENCE',
    'FLAG_EMAIL',
    'TOTAL_MONTHLY_INCOME',             
    'QUANT_BANKING_ACCOUNTS',
    'QUANT_SPECIAL_BANKING_ACCOUNTS',
    'PERSONAL_ASSETS_VALUE',          
    'QUANT_CARS',
    'COMPANY',
    'PROFESSION_CODE',
    'OCCUPATION_TYPE',
    'FLAG_VISA',
    'FLAG_MASTERCARD',
    'FLAG_OTHER_CARDS',           
    'PRODUCT',
    'AGE',
    'RESIDENCIAL_ZIP_3',              
    'HAS_CREDIT_CARD',
    'TARGET_LABEL_BAD=1'                 
    ]
      
    post_data = post_data[selected_features_final]
    return pd.DataFrame(post_data) 

In [17]:
modeling_data_prepared = data_preparation(modeling_data)

C:\Users\santi\AppData\Local\Temp\ipykernel_13488\802532721.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
C:\Users\santi\AppData\Local\Temp\ipykernel_13488\802532721.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
C:\Users\santi\AppData\Local\Temp\ipykernel_13488\802532721.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):


In [19]:
def get_feature_target(
    app_train: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Separates our train and test datasets columns between Features
    (the input to the model) and Targets (what the model has to predict with the
    given features).

    Arguments:
        app_train : pd.DataFrame
            Training datasets
        app_test : pd.DataFrame
            Test datasets

    Returns:
        X_train : pd.DataFrame
            Training features
        y_train : pd.Series
            Training target
        X_test : pd.DataFrame
            Test features
        y_test : pd.Series
            Test target
    """

    # TODO
    # Assign to X_train all the columns from app_train except "TARGET"
    # Assign to y_train the "TARGET" column
    # Assign to X_test all the columns from app_test except "TARGET"
    # Assign to y_test the "TARGET" column
    X_train = app_train.drop(columns=["TARGET_LABEL_BAD=1"])
    y_train = app_train["TARGET_LABEL_BAD=1"]


    return X_train, y_train

In [20]:
def get_train_val_sets(
    X_train: pd.DataFrame, y_train: pd.Series
) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    Split training dataset in two new sets used for train and validation.

    Arguments:
        X_train : pd.DataFrame
            Original training features
        y_train: pd.Series
            Original training labels/target

    Returns:
        X_train : pd.DataFrame
            Training features
        X_val : pd.DataFrame
            Validation features
        y_train : pd.Series
            Training target
        y_val : pd.Series
            Validation target
    """
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, 
        test_size=0.2, 
        random_state=42, 
        shuffle=True    
    )

    return X_train, X_val, y_train, y_val

In [21]:
X_train, y_train = get_feature_target(modeling_data_prepared)
X_train, X_val, y_train, y_val = get_train_val_sets(X_train, y_train)

In [22]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from typing import Tuple, List

def preprocess_data(
    X_train: pd.DataFrame, X_val: pd.DataFrame
) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    print("Input train shape:", X_train.shape)
    print("Input val shape:", X_val.shape, "\n")

    # Copias
    train_df = X_train.copy()
    val_df = X_val.copy()

    # Combinar
    combined_df = pd.concat([train_df, val_df])

    # Reemplazo de strings vacíos
    combined_df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

    # Convertir columnas categóricas a string
    for col in combined_df.select_dtypes(include=["object", "category"]).columns:
        combined_df[col] = combined_df[col].astype(str)

    # Pasar a lowercase las columnas de ciudad
    for col in ["CITY_OF_BIRTH", "RESIDENCIAL_CITY", "RESIDENCIAL_BOROUGH"]:
        if col in combined_df.columns:
            combined_df[col] = combined_df[col].str.lower()

    # Eliminar columnas dominadas, constantes o todo 0
    drop_cols = []
    for col in combined_df.columns:
        try:
            unique_vals = combined_df[col].nunique(dropna=False)
            top_freq = combined_df[col].value_counts(normalize=True, dropna=False).max()
            if unique_vals == 1 or top_freq > 0.98:
                drop_cols.append(col)
            elif (combined_df[col] == 0).sum() == len(combined_df):
                drop_cols.append(col)
        except Exception as e:
            print(f"[WARNING] Skipping column '{col}' during low variance check: {e}")
            continue
    combined_df.drop(columns=drop_cols, inplace=True)

    # Eliminar columnas con más del 60% de NaNs
    nan_ratio = combined_df.isna().mean()
    combined_df.drop(columns=nan_ratio[nan_ratio > 0.6].index, inplace=True)

    # Dividir nuevamente
    n_train = len(train_df)
    train_df = combined_df.iloc[:n_train].copy()
    val_df = combined_df.iloc[n_train:].copy()

    # Clasificación de variables categóricas
    cat_counts = train_df.select_dtypes(include=["object"]).nunique()
    c_ordinal = cat_counts[cat_counts == 2]
    c_onehot = cat_counts[(cat_counts > 2)]

    print(f"[INFO] Ordinal (2 categorías): {list(c_ordinal.index)}")
    print(f"[INFO] One-hot (3-50 categorías): {list(c_onehot.index)}")

    # Encoders
    ordinal_enc = OrdinalEncoder().set_output(transform='pandas')
    onehot_enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore").set_output(transform='pandas')

    if not c_ordinal.empty:
        ordinal_enc.fit(train_df[c_ordinal.index])
    if not c_onehot.empty:
        onehot_enc.fit(train_df[c_onehot.index])

    def encode_df(df):
        x = ordinal_enc.transform(df[c_ordinal.index]) if not c_ordinal.empty else pd.DataFrame()
        y = onehot_enc.transform(df[c_onehot.index]) if not c_onehot.empty else pd.DataFrame()
        df = df.drop(columns=c_ordinal.index.union(c_onehot.index))
        return pd.concat([df, x, y], axis=1)

    train_df = encode_df(train_df)
    val_df = encode_df(val_df)

    # Imputación por mediana
    imp_median = SimpleImputer(strategy='median').set_output(transform='pandas')
    imp_median.fit(train_df.select_dtypes(include=["float64", "int64"]))
    for df in [train_df, val_df]:
        num_cols = df.select_dtypes(include=["float64", "int64"]).columns
        df[num_cols] = imp_median.transform(df[num_cols])

    # Escalado MinMax
    scaler = MinMaxScaler().set_output(transform="pandas")
    scaler.fit(train_df.select_dtypes(include=["float64", "int64"]))
    for df in [train_df, val_df]:
        num_cols = df.select_dtypes(include=["float64", "int64"]).columns
        df[num_cols] = scaler.transform(df[num_cols])

    # Guardar nombres de columnas
    feature_names = train_df.columns.tolist()


    return train_df.to_numpy(), val_df.to_numpy(), feature_names

In [23]:
train_data, val_data, feature_names = preprocess_data(X_train, X_val)

Input train shape: (39998, 28)
Input val shape: (10000, 28) 

[INFO] Ordinal (2 categorías): ['FLAG_RESIDENCIAL_PHONE', 'COMPANY']
[INFO] One-hot (3-50 categorías): ['APPLICATION_SUBMISSION_TYPE', 'SEX', 'STATE_OF_BIRTH', 'CITY_OF_BIRTH', 'RESIDENCIAL_STATE', 'RESIDENCIAL_CITY', 'RESIDENCIAL_ZIP_3']


In [41]:
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import recall_score

best_model = XGBClassifier(
    subsample=0.9,
    scale_pos_weight=np.float64(3.3860500668832407),
    reg_lambda=1,
    reg_alpha=5,
    objective='binary:logistic',
    n_estimators=500,
    min_child_weight=10,
    max_depth=6,
    max_delta_step=0,
    learning_rate=0.02,
    gamma=0.1,
    colsample_bytree=1.0,
    booster='gbtree',
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

best_model.fit(train_data, y_train)

c:\Users\santi\Desktop\FINALRISK\Creda2\Creda\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:24:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=1.0, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.02, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=0,
              max_depth=6, max_leaves=None, min_child_weight=10, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, ...)

WE ARE FOCUSING ON RECALL

In [42]:
# Predecir
y_pred = best_model.predict(train_data)
y_pred_val = best_model.predict(val_data)

# Evaluar
print("Accuracy TRAIN:", accuracy_score(y_train, y_pred))
print("Confusion Matrix TRAIN:\n", confusion_matrix(y_train, y_pred))
print("Classification Report TRAIN:\n", classification_report(y_train, y_pred))
print("RS Train", recall_score(y_train, y_pred))

# Evaluar
print("Accuracy VAL:", accuracy_score(y_val, y_pred_val))
print("Confusion Matrix VAL:\n", confusion_matrix(y_val, y_pred_val))
print("Classification Report VAL:\n", classification_report(y_val, y_pred_val))
print("RS Val", recall_score(y_val, y_pred_val))

Accuracy TRAIN: 0.5667283364168209
Confusion Matrix TRAIN:
 [[14155 15377]
 [ 1953  8513]]
Classification Report TRAIN:
               precision    recall  f1-score   support

           0       0.88      0.48      0.62     29532
           1       0.36      0.81      0.50     10466

    accuracy                           0.57     39998
   macro avg       0.62      0.65      0.56     39998
weighted avg       0.74      0.57      0.59     39998

RS Train 0.8133957576915727
Accuracy VAL: 0.5342
Confusion Matrix VAL:
 [[3404 4022]
 [ 636 1938]]
Classification Report VAL:
               precision    recall  f1-score   support

           0       0.84      0.46      0.59      7426
           1       0.33      0.75      0.45      2574

    accuracy                           0.53     10000
   macro avg       0.58      0.61      0.52     10000
weighted avg       0.71      0.53      0.56     10000

RS Val 0.752913752913753
